# SE 方程 Python 求解（仿 Fortran 版本）

这个笔记本对应你给的 `SE-solver.f90`，完成以下步骤：

1. 读取 Fortran 写出的 `SE-A.ieee` ~ `SE-F.ieee` 与 `SE-rho.ieee`。
2. 用 SOR 方法求解椭圆方程（未知量为流函数 $\psi$）。
3. 由 $\psi$ 诊断二次环流径向风 `U_se` 和垂直风 `W_se`。
4. 作图检查结果并保存输出。

In [ ]:
from pathlib import Path
import struct
import numpy as np
import matplotlib.pyplot as plt

# ========== 网格与物理参数（与 Fortran 一致） ==========
g = 9.806
Dr = 2000.0
Dz = 500.0
Nr = 121
Nz = 37

# 系数文件所在目录（默认是当前工作目录）
DATA_DIR = Path('.')

R = np.arange(Nr, dtype=np.float64) * Dr  # R[0]=0, 对应轴心

In [ ]:
def read_fortran_unformatted_real32(path: Path, shape):
    """
    读取 Fortran sequential unformatted 单记录 real*4 数组。
    适配 write(unit) array 的常见格式（前后4字节记录长度标记）。
    """
    with open(path, 'rb') as f:
        head = f.read(4)
        if len(head) != 4:
            raise IOError(f'文件过短: {path}')
        (nbytes,) = struct.unpack('<i', head)

        payload = f.read(nbytes)
        if len(payload) != nbytes:
            raise IOError(f'记录体长度不足: {path}')

        tail = f.read(4)
        if len(tail) != 4:
            raise IOError(f'缺少记录尾标记: {path}')
        (nbytes_tail,) = struct.unpack('<i', tail)

        if nbytes_tail != nbytes:
            raise IOError(f'记录前后长度不一致: {path}')

    arr = np.frombuffer(payload, dtype='<f4')
    expect = int(np.prod(shape))
    if arr.size != expect:
        raise ValueError(
            f'{path.name} 元素数量不匹配: 读到 {arr.size}, 期望 {expect} (shape={shape})'
        )

    # Fortran 列主序
    return arr.reshape(shape, order='F').astype(np.float64)


def load_coefficients(data_dir: Path):
    A = read_fortran_unformatted_real32(data_dir / 'SE-A.ieee', (Nr, Nz))
    B = read_fortran_unformatted_real32(data_dir / 'SE-B.ieee', (Nr, Nz))
    C = read_fortran_unformatted_real32(data_dir / 'SE-C.ieee', (Nr, Nz))
    D = read_fortran_unformatted_real32(data_dir / 'SE-D.ieee', (Nr, Nz))
    E = read_fortran_unformatted_real32(data_dir / 'SE-E.ieee', (Nr, Nz))
    F = read_fortran_unformatted_real32(data_dir / 'SE-F.ieee', (Nr, Nz))
    rho = read_fortran_unformatted_real32(data_dir / 'SE-rho.ieee', (Nr, Nz + 2))

    return A, B, C, D, E, F, rho

In [ ]:
def sor_solve(A, B, C, D, E, F, dx, dy, max_iter=60000, omega=1.8, tol=1e-16, verbose_every=200):
    """
    求解: A*P_xx + B*P_xy + C*P_yy + D*P_x + E*P_y = F
    网格排布与 Fortran 对齐: P.shape = (Nr, Nz+2), j=0..Nz+1
    """
    nx, ny = A.shape
    P = np.zeros((nx, ny + 2), dtype=np.float64)

    for it in range(1, max_iter + 1):
        max_res = 0.0

        # Fortran: i=2..Nx-1, j=1..Ny
        for i in range(1, nx - 1):
            for j in range(1, ny + 1):
                if i == nx - 2:
                    p_xx = (P[i, j] + P[i - 1, j] - 2.0 * P[i, j]) / dx**2
                    p_xy = (P[i, j + 1] - P[i, j - 1] - P[i - 1, j + 1] + P[i - 1, j - 1]) / (4.0 * dx * dy)
                    p_x = (P[i, j] - P[i - 1, j]) / (2.0 * dx)
                else:
                    p_xx = (P[i + 1, j] + P[i - 1, j] - 2.0 * P[i, j]) / dx**2
                    p_xy = (P[i + 1, j + 1] - P[i + 1, j - 1] - P[i - 1, j + 1] + P[i - 1, j - 1]) / (4.0 * dx * dy)
                    p_x = (P[i + 1, j] - P[i - 1, j]) / (2.0 * dx)

                p_yy = (P[i, j + 1] + P[i, j - 1] - 2.0 * P[i, j]) / dy**2
                p_y = (P[i, j + 1] - P[i, j - 1]) / (2.0 * dy)

                residual = A[i, j - 1] * p_xx + B[i, j - 1] * p_xy + C[i, j - 1] * p_yy + D[i, j - 1] * p_x + E[i, j - 1] * p_y - F[i, j - 1]
                eij = -2.0 * A[i, j - 1] / dx**2 - 2.0 * C[i, j - 1] / dy**2

                if abs(eij) < 1e-30:
                    continue

                P[i, j] = P[i, j] - omega * residual / eij
                max_res = max(max_res, abs(residual))

        # Fortran 边界: P(Nx,j)=P(Nx-1,j)
        P[-1, :] = P[-2, :]

        if verbose_every and (it % verbose_every == 0 or it == 1):
            print(f'iter={it:6d}, max_res={max_res:.3e}')

        if max_res < tol:
            print(f'Converged at iter={it}, max_res={max_res:.3e}')
            break
    else:
        print(f'Not converged after {max_iter} iterations, final max_res={max_res:.3e}')

    return P


def psi_to_uw(psi, rho, R, dr, dz):
    """
    按 Fortran 公式从 psi 诊断 U_se/W_se。
    psi, rho: (Nr, Nz+2)
    """
    nr, nzp2 = psi.shape
    nz = nzp2 - 2

    U = np.zeros_like(psi)
    W = np.zeros_like(psi)

    # W: iz=1..Nz, ir=2..Nr-1 (Fortran 1-based)
    for iz in range(1, nz + 1):
        for ir in range(1, nr - 1):
            denom = 2.0 * dr * R[ir] * rho[ir, iz]
            if abs(denom) > 0.0:
                W[ir, iz] = (psi[ir + 1, iz] - psi[ir - 1, iz]) / denom
        W[0, iz] = W[1, iz]
        W[-1, iz] = W[-2, iz]

    W[:, 0] = 0.0
    W[:, -1] = 0.0

    # U: ir=2..Nr, iz=1..Nz (Fortran 1-based)
    for ir in range(1, nr):
        denom0 = dz * R[ir] * 0.5 * (rho[ir, 0] + rho[ir, 1])
        denom1 = dz * R[ir] * 0.5 * (rho[ir, -1] + rho[ir, -2])
        if abs(denom0) > 0.0:
            U[ir, 0] = -(psi[ir, 1] - psi[ir, 0]) / denom0
        if abs(denom1) > 0.0:
            U[ir, -1] = -(psi[ir, -1] - psi[ir, -2]) / denom1

        for iz in range(1, nz + 1):
            denom = 2.0 * dz * R[ir] * rho[ir, iz]
            if abs(denom) > 0.0:
                U[ir, iz] = -(psi[ir, iz + 1] - psi[ir, iz - 1]) / denom

    U[0, :] = 0.0
    return U, W

In [ ]:
# 1) 读入系数
A, B, C, D, E, Ff, rho = load_coefficients(DATA_DIR)
print('A shape:', A.shape, 'rho shape:', rho.shape)

# 2) 求解 psi
psi = sor_solve(A, B, C, D, E, Ff, dx=Dr, dy=Dz, max_iter=60000, omega=1.8, tol=1e-16, verbose_every=500)

# 3) 诊断 U/W
U_se, W_se = psi_to_uw(psi, rho, R, dr=Dr, dz=Dz)

print('psi range:', float(np.nanmin(psi)), float(np.nanmax(psi)))
print('U_se range:', float(np.nanmin(U_se)), float(np.nanmax(U_se)))
print('W_se range:', float(np.nanmin(W_se)), float(np.nanmax(W_se)))

In [ ]:
# 简单可视化：取内部层（j=1..Nz）
r_km = R / 1000.0
z_km = np.arange(Nz + 2) * Dz / 1000.0

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), constrained_layout=True)

im0 = axes[0].contourf(r_km, z_km, psi.T, levels=31, cmap='RdBu_r')
axes[0].set_title('psi')
axes[0].set_xlabel('Radius (km)')
axes[0].set_ylabel('Height (km)')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].contourf(r_km, z_km, U_se.T, levels=31, cmap='RdBu_r')
axes[1].set_title('U_se (radial)')
axes[1].set_xlabel('Radius (km)')
axes[1].set_ylabel('Height (km)')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].contourf(r_km, z_km, W_se.T, levels=31, cmap='RdBu_r')
axes[2].set_title('W_se (vertical)')
axes[2].set_xlabel('Radius (km)')
axes[2].set_ylabel('Height (km)')
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.show()

In [ ]:
# 保存输出（推荐 .npy，便于后续 Python 处理）
out_dir = Path('se_python_output')
out_dir.mkdir(exist_ok=True)

np.save(out_dir / 'psi.npy', psi)
np.save(out_dir / 'U_se.npy', U_se)
np.save(out_dir / 'W_se.npy', W_se)

print('Saved to:', out_dir.resolve())

## 端到端 SE 诊断管线（3D 场到 SE 解）

下面单元调用 `se_diagnostic_pipeline.py`，完成：

1. 3D CM1 风场/热力场方位平均；
2. 基于切向风热成风反算平衡位温；
3. 构建 SE 六系数 `A,B,C,D,E,F`；
4. 椭圆性检查与惯性稳定度正则化；
5. 强迫项分解绘图（总强迫/热力强迫/动量源强迫）；
6. SOR 求解流函数并诊断 `U_se/W_se`。

In [ ]:
from se_diagnostic_pipeline import PipelineConfig, SourceMaskConfig, run_pipeline

cfg = PipelineConfig(
    input_file='dataset/cm1out.nc',
    output_dir='se_pipeline_output',
    time_index=0,
    # 若模型中热力源/动量源变量名不同，请在这里改名
    q_name='Q',
    fnu_name='Fnu',
    # 椭圆性正则化：添加量级约为惯性稳定度最大值 1e-3 的正增量
    inertia_eps_ratio=1e-3,
    # 接口：可在指定区域关闭热力源/动量源
    source_mask=SourceMaskConfig(
        thermal_scale=1.0,
        momentum_scale=1.0,
        thermal_zero_boxes=[
            # 示例：关闭 0-40 km, 0-3 km 的热力源
            # {'r_min_km': 0.0, 'r_max_km': 40.0, 'z_min_km': 0.0, 'z_max_km': 3.0},
        ],
        momentum_zero_boxes=[
            # 示例：关闭 100-200 km, 0-5 km 的动量源
            # {'r_min_km': 100.0, 'r_max_km': 200.0, 'z_min_km': 0.0, 'z_max_km': 5.0},
        ],
    ),
)

run_pipeline(cfg)
print('完成。结果见目录: se_pipeline_output')